# Chapter 5: Automating Data Analysis

## 1. Introduction

Welcome to automating your R scripts. In this section, you will learn how to pass arguments to an R script directly from the command line, transforming your static code into a flexible and powerful tool. This skill is fundamental for scaling your analyses. Imagine Dr. Chen at City Medical Center needs to generate updated vital signs plots for hundreds of patient data files; instead of manually editing her script for each one, she can write a simple loop in her terminal to run the same script with different patient data files as inputs. This lesson will guide you from basic positional arguments to more robust, named arguments, preparing you for large-scale, automated data processing pipelines in a clinical or research setting.

---

## 2. Key Concepts and Definitions

*   **Command-Line Argument**: An input value provided to a script when it is executed from a command line or terminal. In a medical context, this could be a patient ID, a file path to a specific MRI scan, or a threshold value for a biomarker analysis, allowing the same script to be reused for different subjects or conditions.
*   **Positional Argument**: An argument that is identified by its order in the command line. For example, if a script expects `Rscript analyze.R <patient_file> <date>`, the first value after the script name is always interpreted as the patient file and the second as the date. This method is simple but rigid, much like filling out a paper form where each box has a fixed meaning.
*   **Flag (or Named Argument)**: An argument preceded by a label, such as `--input` or `-i`. This makes the order of arguments irrelevant and the script's usage clearer. For instance, `Rscript analyze.R --date "2024-10-28" --patient_file "p123.csv"` is more readable and less error-prone than relying on position, similar to how a patient's chart has clearly labeled fields like "Patient ID" and "Date of Birth."
*   **`commandArgs(trailingOnly = TRUE)`**: A base R function that captures all command-line arguments that come *after* the script name itself. It returns them as a character vector, which is the simplest way to access inputs without external packages.
*   **`argparse`**: A powerful R package that provides a structured way to define, parse, and validate command-line arguments. It allows for creating required arguments, optional flags, default values, and automatic help messages, making scripts more robust and user-friendly.

---

## 3. Main Content

### 3.1. The Basic Method: Capturing Arguments with Base R

The most direct way to handle command-line inputs in R is with `commandArgs(trailingOnly = TRUE)`. This function captures all inputs provided after the script name and stores them as a character vector. This approach is best for simple scripts with a small number of inputs where the order is fixed.

```R
# generate_plot.R
library(ggplot2)

args <- commandArgs(trailingOnly = TRUE)
if (length(args) != 3) {
  stop("Usage: Rscript generate_plot.R <path/to/data.csv> <'Plot Title'> <line_width>", call. = FALSE)
}

input_file <- args[1]
plot_title <- args[2]
line_width <- as.numeric(args[3])

patient_data <- read.csv(input_file)
vitals_plot <- ggplot(patient_data, aes(x = Day, y = systolic_bp)) +
  geom_line(linewidth = line_width) +
  labs(title = plot_title, y = "Systolic BP (mmHg)")
```

> **Debug Note:** A common error is forgetting to convert character arguments to the correct data type. If you pass `1.2` for `line_width` but forget `as.numeric()`, `ggplot` will throw an error because `linewidth` expects a numeric value, not the character string `"1.2"`. Always validate and convert inputs early in your script.

> **Pro Tip:** Using `call. = FALSE` within the `stop()` function is a best practice for user-facing scripts. It prevents R from printing a complex call stack, providing the user with a clean, direct error message that is easier to understand.

> **Important:** When running an R script from the command line, a `ggplot` object must be explicitly wrapped in `print()` to be rendered. Unlike an interactive R session, the script will not automatically display the last created object. Without `print(vitals_plot)`, your script will run without errors but produce no output.
```R
print(vitals_plot)
```

To run this script from your terminal:
```bash
Rscript generate_plot.R "data/PT000123.csv" "Vitals for Patient 123" 1.2
```

### 3.2. The Advanced Method: The `argparse` Package

> **Medical Background:** Think of the `argparse` package as creating a standardized intake form for your R script. Instead of relying on the order of inputs (like the base R method), you create clearly labeled fields like `--input` or `--bp-threshold`. This is like a patient's chart, where "Patient ID" and "Blood Pressure" are distinct fields, reducing ambiguity and making the script easier and safer for others to use.

For more complex scripts, the `argparse` package is the industry standard. It allows you to create labeled, self-documenting arguments (flags) like `--input` or `--bp-threshold`.

```R
# parser_example.R
library(argparse)
library(ggplot2)

parser <- ArgumentParser(description='Process patient data and plot vitals.')
parser$add_argument("--input", required=TRUE, help="Path to input CSV file")
parser$add_argument("--bp-threshold", type="double", default=140.0, help="Systolic BP threshold (mmHg)")
parser$add_argument("--line_width", type="double", default=1.0, help="Width for the plot line.")
parser$add_argument("--title", default="Patient Vitals", help="Optional plot title")
args <- parser$parse_args()

patient_data <- read.csv(args$input)

vitals_plot <- ggplot(patient_data, aes(x = Day, y = systolic_bp)) +
  geom_line(linewidth = args$line_width) +
  geom_hline(yintercept = args$bp_threshold, color = "red", linetype = "dashed") +
  labs(title = args$title, y = "Systolic BP (mmHg)")

print(vitals_plot)
```

To see the auto-generated help message, run this in your terminal:
```bash
Rscript parser_example.R --help
```
```text
# Expected Output
usage: parser_example.R [-h] --input INPUT [--bp-threshold BP_THRESHOLD]
                        [--line_width LINE_WIDTH] [--title TITLE]

Process patient data and plot vitals.

options:
  -h, --help            show this help message and exit
  --input INPUT         Path to input CSV file
  --bp-threshold BP_THRESHOLD
                        Systolic BP threshold (mmHg)
  --line_width LINE_WIDTH
                        Width for the plot line.
  --title TITLE         Optional plot title
```

To run the script with required and optional arguments:
```bash
Rscript parser_example.R --input "data/PT000123.csv" --bp-threshold 150 --title "BP for Patient 123"
```

---

## 4. Practice Exercises

### Exercise 1: Basic Positional Arguments Basic

**Objective:** Modify a script to accept positional arguments for the input file and plot color.
**Time:** 5 minutes
**Medical Context:** Dr. Chen wants to enhance the plotting script. She needs the ability to quickly change the line color to highlight specific patient charts during presentations.

Modify the base R `generate_plot.R` script logic and save it as `plot_vitals.R`.
1.  The script must accept exactly two command-line arguments: the path to a patient's CSV data file and a color for the plot's line graph (e.g., "blue", "red").
2.  If the number of arguments is not two, stop the script with the message: `Usage: Rscript plot_vitals.R <path/to/data.csv> <color>`.
3.  Use the first argument to read the data and the second to set the color of the `geom_line()`.
4.  Generate and print the plot.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```R
# plot_vitals.R
library(ggplot2)

args <- commandArgs(trailingOnly = TRUE)
if (length(args) != 2) {
  stop("Usage: Rscript plot_vitals.R <path/to/data.csv> <color>", call. = FALSE)
}

input_file <- args[1]
line_color <- args[2]

patient_data <- read.csv(input_file)
vitals_plot <- ggplot(patient_data, aes(x = Day, y = systolic_bp)) +
  geom_line(color = line_color, linewidth = 1) +
  labs(title = paste("Vitals for", input_file), y = "Systolic BP (mmHg)")

print(vitals_plot)
```

**Explanation:** The script captures arguments and checks if there are exactly two. It then assigns the first argument to `input_file` and the second to `line_color`. The `line_color` variable is passed directly to the `color` aesthetic in `geom_line()`.
**Key Learning:** Positional arguments are accessed sequentially from the `args` vector. Simple validation using `length(args)` is crucial for ensuring the script receives the correct inputs.

</div>
</details>

### Exercise 2: Using Named Arguments with `argparse` Intermediate

**Objective:** Practice running a script that uses named arguments and overriding a default value.
**Time:** 5 minutes
**Medical Context:** A clinician needs to generate a plot for a patient with severe hypertension and wants to set a custom alert threshold on the graph, different from the standard 140 mmHg.

Using the `parser_example.R` script from the lesson, write the full terminal command to run the script for the file `data/PT000456.csv` and set the blood pressure threshold to `160.0`.

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


```bash
Rscript parser_example.R --input "data/PT000456.csv" --bp-threshold 160.0
```

**Explanation:** We provide the required `--input` argument with the path to the patient's data file. We then use the optional `--bp-threshold` flag to override the default value of 140.0 with our custom value of 160.0. Since flags are named, the order does not matter.
**Key Learning:** Named arguments make scripts more readable and flexible, allowing users to override default parameters easily.

</div>
</details>

### Exercise 3: Understanding a Script with `--help` Intermediate

**Objective:** Use the auto-generated help message from an `argparse` script to understand its functionality.
**Time:** 5 minutes
**Medical Context:** A new researcher joins Dr. Chen's team and is given an R script named `analyze_cohort.R`. Before running it, they need to understand what inputs it requires and what options are available, without reading the source code.

Assume `parser_example.R` is the `analyze_cohort.R` script. Write the command to display its help message and answer the following questions based on the output:
1.  Which argument is mandatory?
2.  What is the default value for the `--title` argument?

**📝 Your Solution:**

In [ ]:
# TODO: Write your solution here
# Hint: Check the task description above




<details>
<summary style="background-color: #f0f0f0; padding: 10px; cursor: pointer; border-radius: 5px;">
<strong>🔍 Click to reveal solution</strong>
</summary>

<div style="padding: 10px; border-left: 3px solid #2196F3; margin-top: 10px;">

**Solution**


**Command:**
```bash
Rscript parser_example.R --help
```

**Answers:**
1.  The `--input` argument is mandatory. In the help text, arguments that are not enclosed in square brackets `[]` in the usage line are required.
2.  The default value for the `--title` argument is `"Patient Vitals"`. This is explicitly stated in the help text for that option.

**Explanation:** The `--help` flag is automatically created by `argparse` and provides a user-friendly summary of all defined arguments, their purpose (from the `help` text), their types, and their default values.
**Key Learning:** Well-documented scripts using `argparse` are self-explanatory, which is crucial for collaboration and for building maintainable analysis pipelines.

</div>
</details>

---

## 5. Practical Applications

*   **Batch Processing of Clinical Trial Data**: In a multi-site clinical trial, data for thousands of patients arrives in separate CSV files. A single R script can be written to perform a standardized analysis (e.g., calculate response to treatment). Using a command-line argument for the input file path (`--input`), a simple shell script can loop through all files, execute the R script for each one, and save the results, ensuring consistency and saving hundreds of hours of manual work.
*   **Automated Quality Control (QC) Reporting**: When analyzing high-throughput sequencing data (e.g., from a genomics core), it's critical to run QC checks. An R script using `argparse` can be designed to take a data file path, a set of QC thresholds (`--min_reads`, `--max_error`), and an output directory (`--outdir`). This script can be integrated into an automated pipeline that runs immediately after the sequencing instrument finishes, generating a PDF report with plots and tables that flag any low-quality samples for review.
*   **Dynamic Epidemiological Modeling**: Researchers modeling the spread of a disease can use command-line arguments to tune model parameters without editing code. Arguments like `--infection_rate 0.8` or `--vaccine_efficacy 0.92` can be passed to an R script that runs a simulation. This allows for rapid "what-if" scenario testing, where hundreds of parameter combinations can be explored systematically on a high-performance computing cluster to find the most likely outcomes.

---

## 6. Summary and Key Takeaways

In this section, we've explored how to make R scripts dynamic and reusable by accepting command-line arguments. We covered the straightforward base R method using `commandArgs()` for simple cases and the more powerful, flexible `argparse` package for building complex, user-friendly command-line tools. Mastering this skill is a critical step toward automating repetitive analyses and building scalable data processing pipelines.

*   **Base R is for Simplicity**: `commandArgs(trailingOnly = TRUE)` is effective for scripts with a few, fixed-order inputs. Remember to validate the number of arguments and convert data types manually.
*   **`argparse` is for Robustness**: For scripts with multiple, optional, or named inputs, `argparse` provides validation, default values, type conversion, and automatic help messages, making your tools easier for you and others to use.
*   **Automation is the Goal**: The primary purpose of using command-line arguments is to enable automation. Your script becomes a building block that can be called repeatedly by other programs to process large volumes of data without human intervention.
*   **`print()` is Essential**: When running a script from the command line, `ggplot` objects must be explicitly passed to `print()` or a save function like `ggsave()` to be rendered.

> **Reflection Moment:** You've now seen two ways to pass arguments to a script. The base R method is quick for simple, personal scripts. The `argparse` method is more robust and user-friendly. When might the extra setup time for `argparse` be justified in a clinical research or operational setting? Consider factors like script complexity, number of users, and the potential impact of input errors.

Next, we will focus on programmatically saving your outputs, such as plots and data tables, to files, completing the workflow for a fully automated analysis pipeline.

---